# Route B (fly): download single-cell ATAC files from the Aerts lab

Run Cells 1 and 2 to see what is available. Then type the runs you want into **RUNS** in Cell 1, run Cell 1 again, and run Cell 3. The files go into the folder the main notebook reads.

In [ ]:
#@title Cell 1 - Settings and connect Google Drive
import os, re, subprocess, urllib.parse
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

BASE_URL = "https://flybrain.aertslab.org/downloads/cellRanger/"  #@param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/route_b_fly"  #@param {type:"string"}
#@markdown Leave **RUNS** empty the first time: Cell 2 lists what is available. Then type the run names you want
#@markdown (comma-separated) and run Cell 3.
RUNS = ""  #@param {type:"string"}
DATA_DIR = os.environ.get("ROUTE_B_DATA", DATA_DIR)
if DATA_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(DATA_DIR, exist_ok=True)


class StopHere(Exception):
    """Stops the notebook with a plain-language message."""


WANTED = ("fragments.tsv.gz", "singlecell.csv")
print("Listing:", BASE_URL, "| saving to:", DATA_DIR)

In [ ]:
#@title Cell 2 - List the runs on the Aerts lab site (about a minute)
def listing(url):
    r = subprocess.run(["curl", "-fsSL", "-m", "60", url], capture_output=True, text=True)
    if r.returncode != 0:
        return None, r.stderr.strip()[:200]
    links = re.findall(r'href="([^"?#]+)"', r.stdout)
    out = []
    for h in links:
        full = urllib.parse.urljoin(url, h)
        if full.startswith(url) and full != url:
            out.append(full)
    return sorted(set(out)), ""


def size_of(url):
    r = subprocess.run(["curl", "-fsSLI", "-m", "60", url], capture_output=True, text=True)
    m = re.findall(r"(?i)content-length:\s*(\d+)", r.stdout)
    return int(m[-1]) / 1e6 if m else float("nan")


top, err = listing(BASE_URL)
if top is None:
    raise StopHere(f"Could not open {BASE_URL} ({err}). Check the address opens in your browser, then run again.")
found, queue, seen = [], [(u, 0) for u in top], set()
while queue:
    u, depth = queue.pop(0)
    if u in seen:
        continue
    seen.add(u)
    name = u.rstrip("/").split("/")[-1]
    if not u.endswith("/"):
        if name in WANTED or name.endswith(("fragments.tsv.gz", "singlecell.csv")):
            found.append(u)
        continue
    if depth < 4:
        sub, _ = listing(u)
        queue += [(s, depth + 1) for s in (sub or [])]

rows = []
for u in found:
    rel = u[len(BASE_URL):]
    folder = re.sub(r"(^|/)outs$", "", os.path.dirname(rel)).strip("/")   # one sequencing run per folder
    run = folder.replace("/", "_") or "top"
    rows.append({"run": run, "file": rel, "MB": round(size_of(u))})
files = pd.DataFrame(rows)
if files.empty:
    print("Top-level entries:", [t[len(BASE_URL):] for t in top][:60])
    raise StopHere("No fragments.tsv.gz or singlecell.csv files found under this address. Send me the list above.")
pd.set_option("display.max_rows", 500)
display(files)
summary = files.groupby("run").agg(files=("file", "size"), MB=("MB", "sum"))
display(summary)
print("\nRuns that have both files:", [r for r, g in files.groupby("run")
                                       if any(f.endswith("fragments.tsv.gz") for f in g.file)
                                       and any(f.endswith("singlecell.csv") for f in g.file)])
print("Type the runs you want into RUNS in Cell 1 (comma-separated), run Cell 1 again, then Cell 3.")

In [ ]:
#@title Cell 3 - Download the chosen runs into your Drive folder
chosen = [r.strip().strip("/") for r in RUNS.split(",") if r.strip()]
if not chosen:
    raise StopHere("RUNS in Cell 1 is empty. Type run names from the Cell 2 list, run Cell 1 again, then this cell.")
unknown = [r for r in chosen if r not in set(files.run)]
if unknown:
    raise StopHere(f"Not in the Cell 2 list: {unknown}. Copy the names exactly.")
need = files[files.run.isin(chosen)]
print(f"About {need.MB.sum() / 1000:.1f} GB to download. Check your Google Drive has that much free space.")
for run, g in need.groupby("run"):
    os.makedirs(os.path.join(DATA_DIR, run), exist_ok=True)
    for name in WANTED:
        hits = [f for f in g.file if f.endswith(name)]
        if not hits:
            print(f"   {run}: no {name} on the site")
            continue
        url = BASE_URL + hits[0]
        dest = os.path.join(DATA_DIR, run, name)
        if os.path.exists(dest) and os.path.getsize(dest) > 0:
            print(f"   {run}/{name}: already downloaded")
            continue
        r = subprocess.run(["curl", "-fsSL", "-C", "-", "-m", "7200", "-o", dest + ".part", url],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(f"   {run}/{name}: FAILED ({r.stderr.strip()[:150]}). Run this cell again to resume.")
            continue
        os.replace(dest + ".part", dest)
        print(f"   {run}/{name}: {os.path.getsize(dest) / 1e6:.0f} MB")
print("\nDone. In the main notebook (RouteB_fly_ploidy_redox), set RUNS in Cell 1 to:", ", ".join(chosen))